In [1]:
from pathlib import Path
import re
from collections import Counter, defaultdict
from bs4 import BeautifulSoup
import sys


PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.config import SPLITS

In [2]:
# Parent labels
PARENT_LABELS = [
    "decision",
    "legislation",
    "secondary sources",
    "unable to classify"
]

# Child/sub labels
SUBLABELS = [
    "authors",
    "title",
    "fragment",
    "citation",
    "source",
]

In [6]:
per_split_totals = defaultdict(lambda: {
    "parent": 0,
    "child": 0,
    "total": 0,
})

for split in SPLITS:
    split_dir = PROJECT_ROOT / "data" / "annotated" / split

    if not split_dir.exists():
        print(f"skip (not found): {split_dir}")
        continue

    files = sorted(split_dir.rglob("*.html"))

    if not files:
        print(f"no html files in: {split_dir}")
        continue

    print(f"--- split: {split} (files: {len(files)}) ---")

    for f in files:
        try:
            text = f.read_text(encoding="utf-8", errors="ignore")
            soup = BeautifulSoup(text, "html.parser")

            # Find all manual_label and auto_label tags
            tags = soup.find_all(["manual_label", "auto_label"])

            parent_count = 0
            child_count = 0

            for tag in tags:
                label = tag.get("labelname")

                if not label:
                    continue

                if label in PARENT_LABELS:
                    parent_count += 1
                elif label in SUBLABELS:
                    child_count += 1

            total = parent_count + child_count

            per_split_totals[split]["parent"] += parent_count
            per_split_totals[split]["child"] += child_count
            per_split_totals[split]["total"] += total

            rel = f.relative_to(PROJECT_ROOT)
            print(
                f"{rel}: "
                f"parents={parent_count}, "
                f"children={child_count}, "
                f"total={total}"
            )

        except Exception as e:
            print(f"error processing {f}: {e}")

    print()

print("\n--- per-split totals ---")

for split in SPLITS:
    if split in per_split_totals:
        counts = per_split_totals[split]

        print(
            f"{split}: "
            f"parents={counts['parent']}, "
            f"children={counts['child']}, "
            f"total={counts['total']}"
        )

--- split: train (files: 4) ---
data\annotated\train\1994CanLII4528NLCA.html: parents=167, children=257, total=424
data\annotated\train\1997CanLII16226ONCA.html: parents=623, children=1288, total=1911
data\annotated\train\2008CSC9.html: parents=484, children=915, total=1399
data\annotated\train\2019SCC65.html: parents=1446, children=3212, total=4658

--- split: test (files: 4) ---
data\annotated\test\2001CanLII21117.html: parents=322, children=467, total=789
data\annotated\test\2002SCC33.html: parents=328, children=625, total=953
data\annotated\test\2005QCCA437.html: parents=117, children=195, total=312
data\annotated\test\2024NBKB203.html: parents=244, children=414, total=658

--- split: dev (files: 3) ---
data\annotated\dev\1989CanLII1415ONCA.html: parents=95, children=146, total=241
data\annotated\dev\2016NBOMB12.html: parents=136, children=170, total=306
data\annotated\dev\2021QCCA1675.html: parents=99, children=172, total=271

--- split: incoming (files: 34) ---
data\annotated\inc

In [4]:


counts = Counter()

for split in SPLITS:

    split_dir = PROJECT_ROOT /"data" / "annotated" / split

    html_files = list(split_dir.rglob("*.html"))
    print(len(html_files), "html files found in", split_dir)
    for html_file in html_files:
        try:
            with open(html_file, "r", encoding="utf-8") as f:
                soup = BeautifulSoup(f, "html.parser")

            # Find all manual_label and auto_label tags
            tags = soup.find_all(["manual_label", "auto_label"])

            for tag in tags:
                label = tag.get("labelname")

                if label:
                    counts[label] += 1

        except Exception as e:
            print(f"Error processing {html_file}: {e}")

# =========================
# PRINT FINAL TABLE
# =========================

print("\n" + "=" * 80)
print("TOTAL LABEL COUNTS")
print("=" * 80)

print(f"{'LABEL':<20} {'COUNT':>10}")
print("-" * 32)

# Parent labels
parent_total = 0
for label in PARENT_LABELS:
    c = counts[label]
    parent_total += c
    print(f"{label:<20} {c:>10}")

print("-" * 32)
print(f"{'TOTAL PARENT':<20} {parent_total:>10}")

print()

# Sublabels
sub_total = 0
for label in SUBLABELS:
    c = counts[label]
    sub_total += c
    print(f"{label:<20} {c:>10}")

print("-" * 32)
print(f"{'TOTAL SUBLABELS':<20} {sub_total:>10}")

print()

# Grand total
grand_total = parent_total + sub_total
print(f"{'TOTAL COUNT':<20} {grand_total:>10}")

4 html files found in c:\Users\zakga\OneDrive\Documents\code\LeREaD\data\annotated\train
4 html files found in c:\Users\zakga\OneDrive\Documents\code\LeREaD\data\annotated\test
3 html files found in c:\Users\zakga\OneDrive\Documents\code\LeREaD\data\annotated\dev
34 html files found in c:\Users\zakga\OneDrive\Documents\code\LeREaD\data\annotated\incoming

TOTAL LABEL COUNTS
LABEL                     COUNT
--------------------------------
decision                   5578
legislation                4652
secondary sources           735
unable to classify           91
--------------------------------
TOTAL PARENT              11056

authors                     572
title                      8576
fragment                   6458
citation                   4494
source                      446
--------------------------------
TOTAL SUBLABELS           20546

TOTAL COUNT               31602


### # Word tokens per document

In [5]:
per_split_totals = defaultdict(int)
per_split_word_counts = defaultdict(list)  # Store word counts per file

for split in SPLITS:
    split_dir = PROJECT_ROOT / "data" / "annotated" / split

    if not split_dir.exists():
        print(f"skip (not found): {split_dir}")
        continue

    files = sorted(split_dir.rglob("*.html"))

    if not files:
        print(f"no html files in: {split_dir}")
        continue

    print(f"--- split: {split} (files: {len(files)}) ---")

    for f in files:
        try:
            text = f.read_text(encoding="utf-8", errors="ignore")
            soup = BeautifulSoup(text, "html.parser")

            # Extract raw text from the entire HTML
            raw_text = soup.get_text(separator=" ", strip=True)
            word_tokens = len(raw_text.split())

            # Store word count for this file
            rel = f.relative_to(PROJECT_ROOT)
            per_split_word_counts[split].append((rel, word_tokens))

            # Print file and word count
            print(f"{rel}: {word_tokens} word tokens")

        except Exception as e:
            print(f"error processing {f}: {e}")

    print()

# Print fancy summary
print("\n" + "="*60)
print("         WORD TOKEN COUNT SUMMARY         ")
print("="*60)
for split in SPLITS:
    if split in per_split_word_counts:
        print(f"\n--- {split.upper()} ---")
        for rel, count in per_split_word_counts[split]:
            print(f"  {rel}: {count}")
        total = sum(count for _, count in per_split_word_counts[split])
        print(f"  Total for {split}: {total} word tokens")
print("="*60)

--- split: train (files: 4) ---
data\annotated\train\1994CanLII4528NLCA.html: 10144 word tokens
data\annotated\train\1997CanLII16226ONCA.html: 40540 word tokens
data\annotated\train\2008CSC9.html: 28442 word tokens
data\annotated\train\2019SCC65.html: 65440 word tokens

--- split: test (files: 4) ---
data\annotated\test\2001CanLII21117.html: 17021 word tokens
data\annotated\test\2002SCC33.html: 35285 word tokens
data\annotated\test\2005QCCA437.html: 6570 word tokens
data\annotated\test\2024NBKB203.html: 16372 word tokens

--- split: dev (files: 3) ---
data\annotated\dev\1989CanLII1415ONCA.html: 4385 word tokens
data\annotated\dev\2016NBOMB12.html: 7671 word tokens
data\annotated\dev\2021QCCA1675.html: 4773 word tokens

--- split: incoming (files: 34) ---
data\annotated\incoming\1983CanLII2967L.html: 2505 word tokens
data\annotated\incoming\1993CanLII1889PESCTSD.html: 10377 word tokens
data\annotated\incoming\1993CanLII3004FCA.html: 22255 word tokens
data\annotated\incoming\1993NSSC71.h